In [ ]:
from pathlib import Path

HOME_PATH = Path('./').resolve()
cfg_path = HOME_PATH / Path('hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/configs')

In [ ]:
import itertools
import yaml

## manifold stats py -- config generate

In [ ]:
MODULE_PATH = str(HOME_PATH / Path('hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats') / 'manifold_stats.py')

In [ ]:
METHOD_DIMS = {
    "PCA":    [2, 5, 10],
    "TSNE":   [2, 3],
    "ISOMAP": [2, 5, 10]
}

metrics = ["euclidean"]

stats = [
    "residual_variance",
    "trustworthiness_auc",
    "continuity_auc",
    "lcmc_auc",
    "q_local_global",
]

dimreduce_schemes = {}

for method, embedding_dims in METHOD_DIMS.items():
    for dim, metric in itertools.product(embedding_dims, metrics):    
        name = f"{method.lower()}_dim{dim}_{metric}"
        dimreduce_schemes[name] = {
            "method": method,
            "embedding_dim": dim,
            "stats": list(stats),
            "metric": metric,
            "random_state": rs,
        }

In [ ]:
# import itertools
# for method, dim, metric in itertools.product(methods, embedding_dims, metrics):

In [ ]:
cfg = {"ReducedStatistics": {MODULE_PATH: {
    "DimReductionStatistics": {"schemes": dimreduce_schemes},
    }}}

In [ ]:
class LongKeyDumper(yaml.SafeDumper):
    def check_simple_key(self):
        return True

    def ignore_aliases(self, data):
        return True

with open(cfg_path / "manifold_stats.yml", "w") as f:
    yaml.dump(cfg, f, Dumper=LongKeyDumper, sort_keys=False, width=10**6)

## distance distrib stats py -- config generate

In [ ]:

MODULE_PATH = str(HOME_PATH / Path('hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats') / 'distance_distrib_stats.py')

metrics = [
        'canberra', 'chebyshev', 
        'cityblock', 'cosine', 
        'euclidean','minkowski'
    ]
minkowski_ps = [1.5,3,4]
modes = ["WASSERSTEIN_DIST","JS_DIST"]

points = ["global_mean_centroid", "global_median_centroid", "global_trimmed_mean_centroid"]

point_pairs = [
    ("global_mean_centroid", "global_median_centroid"),
    ("global_trimmed_mean_centroid", "global_median_centroid"),
]

# if point_pairs is None:
#     point_pairs = list(itertools.combinations(points, 2))
# point_pairs = list(dict.fromkeys(tuple(sorted(p)) for p in point_pairs if p[0] != p[1]))

centroid_schemes = {}
global_centroid_schemes = {}

for metric in metrics:
    ps = minkowski_ps if metric == "minkowski" else [None]
    
    for p in ps:
        
        for ix, (pt1, pt2) in enumerate(point_pairs):
            name = f"{metric}{'' if p is None else f'_p{p}'}_{pt1}_vs_{pt2}"
            
            centroid_schemes[name] = {
                "metric": metric,
                "point1": pt1,
                "point2": pt2,
                "minkowski_p": p,
                "modes": modes
            }

for ix, (pt1, pt2) in enumerate(point_pairs):
    name = f"{pt1}_vs_{pt2}"    
    global_centroid_schemes[name] = {
        "point1": pt1,
        "point2": pt2,
    }

print(f"{len(centroid_schemes)} schemes")
print(f"{len(global_centroid_schemes)} schemes")

In [ ]:
dd_basic = {}
dd_modes = {}

distance_distrib_stats = [
    'mean','median','IQR','std','skew','kurtosis'
]

for metric in metrics:
    ps = minkowski_ps if metric == "minkowski" else [None]
    
    for p in ps:
        
        for pt in points:
            
            name = f"{metric}{'' if p is None else f'_p{p}'}_{pt}"
            
            dd_basic[name] = {
                "metric": metric,
                "point": pt,
                "minkowski_p": p,
                "stats": list(distance_distrib_stats)
            }

            dd_modes[name] = {
                "metric": metric,
                "point": pt,
                "minkowski_p": p
            }

In [ ]:
print(f"{len(dd_basic)} schemes")
print(f"{len(dd_modes)} schemes")

In [ ]:
cfg = {"ReducedStatistics": {MODULE_PATH: {
    "CentroidDistanceDistributionCompare": {"schemes": centroid_schemes},
    "GlobalCentroidDiffs" : {"schemes" : global_centroid_schemes},
    "DistanceDistributionBasicSummarize" : {"schemes" : dd_basic},
    "DistanceDistributionModesSummarize" : {"schemes" : dd_modes},    
    }}}

In [ ]:
# import pickle
# with open(cfg_path / "centroid_compare.pkl", "wb") as f:
#     pickle.dump(cfg, f)

In [ ]:
# 587//25

In [ ]:
class LongKeyDumper(yaml.SafeDumper):
    def check_simple_key(self):
        return True

    def ignore_aliases(self, data):
        return True

with open(cfg_path / "dist_distrib_stats_metricSweep.yml", "w") as f:
    yaml.dump(cfg, f, Dumper=LongKeyDumper, sort_keys=False, width=10**6)